# Portfolio-safe version

This notebook is a sanitized portfolio adaptation of the author's MSc Data Analytics project.
Environment-specific paths, cloud bucket names, notebook outputs, and exact patient examples
have been removed or generalized. The original methodology and core code structure are preserved.

**Data note:** the underlying TCIA imaging/clinical data are not redistributed in this repository.
Configure your own authorized/local dataset paths before running the notebook.


# CoxPH vs DeepSurv for PFS — Clinical, Radiomics, and Combined (Anti‑leakage, Balanced Group Splits)

- Patient‑level, group‑preserving splits (train/val/test) with optional balancing on **LungMets** prevalence.
- Anti‑leakage filters that drop outcome/recurrence/status/follow‑up/treatment columns.
- Clean CoxPH pipeline (elastic‑net or ridge) with feature pruning/alignment.
- DeepSurv (pycox) MLP with early stopping.
- Radiomics support: per‑patient aggregation (median across studies), standardization, variance filtering, and top‑K selection.
- Repeated evaluations with fresh random splits and final **SUMMARY** blocks:
  - `=== Cox SUMMARY (mean ± sd, Test C-index) ===`
  - `=== DeepSurv SUMMARY (mean ± sd, Test C-index) ===`

In [ ]:
# %%capture
# Install once if needed: pip install lifelines pycox torchtuples pyarrow scikit-survival
import os, re, math, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

import torch
import torchtuples as tt
from pycox.models import CoxPH as DeepCoxPH

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)

## Input data location

The public portfolio version expects preprocessed inputs under `data/processed_splits/`.
The underlying patient-level data are intentionally not included.


In [ ]:
PROCESSED_DIR = os.getenv("PROCESSED_DIR", "data/processed_splits")
os.makedirs(PROCESSED_DIR, exist_ok=True)
print("Using processed data directory:", PROCESSED_DIR)


## Paths & toggles

In [ ]:
PROCESSED_DIR = os.getenv("PROCESSED_DIR", "data/processed_splits")
RADIOMICS_FILE = os.path.join(PROCESSED_DIR, '03_Consolidated_radiomics_all_series.csv')

# Repeats & balancing
REPEATS_R = 30         # number of repeated group-splits
BALANCED_REPEATS = True
BALANCE_TOL = 0.15     # acceptable deviation of LungMets rate from the global rate

# Radiomics feature selection
USE_RADIOMICS = True
VAR_THRESHOLD = 1e-8   # tiny variance cutoff post-standardization
TOPK_RAD = 50          # keep top-K radiomics by variance (after z-score)

# Anti-leakage patterns (drop any column that matches)
LEAKY_PATTERNS = [
    r'(?i)\\boutcome\\b', r'(?i)\\brecurr', r'(?i)\\bmet(s|astasis|astases)\\b', r'(?i)\\bprogress',
    r'(?i)\\bfollow[-_ ]?up', r'(?i)\\bstatus\\b', r'(?i)\\bdeath\\b|\\bdead\\b|\\bdied\\b',
    r'(?i)diagnosis\\s*to\\s*outcome', r'(?i)diagnosis\\s*to\\s*last\\s*follow', r'(?i)^treatment\\b'
]

# Baseline (turn OFF groups you don't want to use)
ALLOW_HISTOLOGY = False
ALLOW_SITE      = False
ALLOW_MSKCC     = False
ALLOW_TREATMENT = False
ALLOW_LUNGMETS  = True   # set False to stress-test robustness


## Load labels & clinical features (with anti-leakage filters)

In [ ]:

import os, re
import pandas as pd

if 'PROCESSED_DIR' not in globals():
    PROCESSED_DIR = os.getenv("PROCESSED_DIR", "data/processed_splits")

# ---- Toggles for baseline groups allowed in the model
ALLOW_HISTOLOGY = False
ALLOW_SITE      = False
ALLOW_MSKCC     = False
ALLOW_TREATMENT = False
ALLOW_LUNGMETS  = True

# ---- Column-name patterns that indicate leakage----
LEAKY_PATTERNS = [
    r'\boutcome\b',
    r'\brecurr',                         # recurrence*
    r'\bmet(s|astasis|astases)\b',
    r'\bprogress',                       # progression*
    r'\bfollow[-_ ]?up',
    r'\bstatus\b',
    r'\bdeath\b|\bdead\b|\bdied\b',
    r'diagnosis\s*to\s*outcome',
    r'diagnosis\s*to\s*last\s*follow',
    r'^treatment\b'
]

def _read_csv(path):
    if not os.path.isfile(path):
        raise FileNotFoundError(path)
    return pd.read_csv(path)

def _rename_patientid(df):
    if 'PatientID' not in df.columns:
        cand = [c for c in df.columns if c.lower() in ('patientid','patient_id','patient','id')]
        if not cand:
            raise ValueError("Couldn't find a PatientID column.")
        df = df.rename(columns={cand[0]: 'PatientID'})
    return df

def _print_dropped(title, cols):
    if not cols:
        return
    print(title)
    for c in cols:
        print(f"  - {c}")

def drop_leaky_cols(X, id_col='PatientID'):
    feats = [c for c in X.columns if c != id_col]
    drop = [c for c in feats if any(re.search(p, c, flags=re.I) for p in LEAKY_PATTERNS)]
    if drop:
        _print_dropped("[Anti-leakage] Dropping %d columns:" % len(drop), drop)
        X = X.drop(columns=drop)
    return X

#
EXCLUDE_GROUPS = []
if not ALLOW_HISTOLOGY: EXCLUDE_GROUPS += [r'^histological type_']
if not ALLOW_SITE:      EXCLUDE_GROUPS += [r'^site of primary sts_']
if not ALLOW_MSKCC:     EXCLUDE_GROUPS += [r'^mskcc type_']
if not ALLOW_TREATMENT: EXCLUDE_GROUPS += [r'^treatment_']
if not ALLOW_LUNGMETS:  EXCLUDE_GROUPS += [r'^lungmets$']

def apply_group_exclusions(X, id_col='PatientID'):
    feats = [c for c in X.columns if c != id_col]
    if not EXCLUDE_GROUPS:
        return X

    combined = re.compile('|'.join(f'(?:{p})' for p in EXCLUDE_GROUPS), flags=re.I)
    s = pd.Series(feats)
    mask = ~s.str.contains(combined)
    kept = s[mask].tolist()
    drop = [c for c in feats if c not in kept]
    if drop:
        _print_dropped("[Baseline filter] Dropping %d columns (groups):" % len(drop), drop)
    return pd.concat([X[[id_col]], X[kept]], axis=1)

# ---- Load labels & clinical matrices ----
labels = _rename_patientid(_read_csv(os.path.join(PROCESSED_DIR, 'survival_labels_by_patient.csv')))
need = {'PatientID','PFS_time_days','PFS_event'}
miss = need - set(labels.columns)
if miss:
    raise ValueError(f"Missing columns in labels: {miss}")

X_clin_cox  = _rename_patientid(_read_csv(os.path.join(PROCESSED_DIR, 'clinical_features_cox_lasso.csv')))
X_clin_deep = _rename_patientid(_read_csv(os.path.join(PROCESSED_DIR, 'clinical_features_deepsurv.csv')))

# ---- Sanitize: anti-leakage + baseline group exclusions ----
X_clin_cox  = apply_group_exclusions(drop_leaky_cols(X_clin_cox))
X_clin_deep = apply_group_exclusions(drop_leaky_cols(X_clin_deep))

# ---- Align patient sets (intersection) & sanity checks ----
pids_all = set(labels.PatientID)
pids_cox = set(X_clin_cox.PatientID)
pids_dep = set(X_clin_deep.PatientID)
pids = sorted(pids_all & pids_cox & pids_dep)
if len(pids) != len(pids_all):
    print(f"[Warning] Restricting to intersection of PatientID: {len(pids)}/{len(pids_all)}")

labels      = labels[labels.PatientID.isin(pids)].sort_values('PatientID').reset_index(drop=True)
X_clin_cox  = X_clin_cox[X_clin_cox.PatientID.isin(pids)].sort_values('PatientID').reset_index(drop=True)
X_clin_deep = X_clin_deep[X_clin_deep.PatientID.isin(pids)].sort_values('PatientID').reset_index(drop=True)

assert labels.PatientID.is_unique,      "Duplicate PatientID in labels."
assert X_clin_cox.PatientID.is_unique,  "Duplicate PatientID in clinical_features_cox_lasso."
assert X_clin_deep.PatientID.is_unique, "Duplicate PatientID in clinical_features_deepsurv."

print('Clinical (Cox) shape:', X_clin_cox.shape)
print('Clinical (DeepSurv) shape:', X_clin_deep.shape)
print('\nLabels preview:')
print(labels[['PatientID','PFS_event','PFS_time_days']].head(3).to_string(index=False))


## Load radiomics (optional) and build per‑patient matrix

In [ ]:
# === Enable & locate radiomics CSV ===
import os, glob
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

# If these globals weren't set yet:
if 'PROCESSED_DIR' not in globals():
    PROCESSED_DIR = os.getenv("PROCESSED_DIR", "data/processed_splits")
os.makedirs(PROCESSED_DIR, exist_ok=True)

# 1) ensure radiomics is enabled
USE_RADIOMICS = True ###

# 2) candidate default path (most common name you shared)
default_rad = os.path.join(PROCESSED_DIR, '03_Consolidated_radiomics_all_series.csv')

# 3) try to reuse an already-defined RADIOMICS_FILE (if valid)
if 'RADIOMICS_FILE' in globals() and os.path.isfile(RADIOMICS_FILE):
    rad_path = RADIOMICS_FILE
elif os.path.isfile(default_rad):
    rad_path = default_rad
else:
    # 4) auto-discover if not found yet
    candidates = []
    # look inside processed_splits
    candidates += glob.glob(os.path.join(PROCESSED_DIR, '*radiomic*.csv'))
    candidates += glob.glob(os.path.join(PROCESSED_DIR, '*Radiomic*.csv'))
    # also look in /content root (in case you copied manually)
    candidates += glob.glob('data/*radiomic*.csv')
    candidates += glob.glob('data/*Radiomic*.csv')

    # if still nothing, try the exact filename anywhere under /content
    if not candidates:
        candidates += glob.glob('data/**/03_Consolidated_radiomics_all_series.csv', recursive=True)

    if candidates:
        # choose the first one (or adjust here to pick a specific one)
        rad_path = sorted(candidates)[0]
    else:
        rad_path = None

# 5) show what we found and set RADIOMICS_FILE for downstream code
if rad_path and os.path.isfile(rad_path):
    RADIOMICS_FILE = rad_path
    print(f"[Radiomics] Using file: {RADIOMICS_FILE}")
else:
    RADIOMICS_FILE = None
    print("[Radiomics] CSV not found. "
          "If it's in Google Drive, copy it to data/processed_splits "
          "or set RADIOMICS_FILE='/path/to/03_Consolidated_radiomics_all_series.csv'.")

# 6) Build the matrix if available (uses your existing function)
if RADIOMICS_FILE:
    X_rad = build_radiomics_matrix(RADIOMICS_FILE)
    display(X_rad.head(3))
else:
    X_rad = None


## Helper: build merged dataset per split

In [ ]:
def build_dataset(X_base, labels_df, patient_ids, X_rad=None):
    df = X_base[X_base['PatientID'].isin(patient_ids)].copy()
    out = df.merge(labels_df[['PatientID','PFS_time_days','PFS_event']], on='PatientID', how='left')
    if X_rad is not None:
        out = out.merge(X_rad, on='PatientID', how='left', suffixes=('', '__rad'))
    return out

def lungmets_rate(pats, Xc):
    if 'LungMets' not in Xc.columns:
        return np.nan
    m = Xc[Xc.PatientID.isin(pats)]['LungMets'].mean()
    return float(m) if pd.notna(m) else np.nan

def split_patients(patient_ids, test_size=0.15, val_size=0.15, rng=None):
    if rng is None: rng = np.random.default_rng()
    pats = np.array(patient_ids)
    gss1 = GroupShuffleSplit(n_splits=1, test_size=test_size+val_size, random_state=int(rng.integers(1e9)))
    tr_idx, hold_idx = next(gss1.split(pats, groups=pats))
    tr, hold = pats[tr_idx], pats[hold_idx]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=test_size/(test_size+val_size), random_state=int(rng.integers(1e9)))
    va_idx, te_idx = next(gss2.split(hold, groups=hold))
    return tr, hold[va_idx], hold[te_idx]

def balanced_split(labels_df, Xc, tol=0.15, max_tries=2000, rng=None):
    rng = rng or np.random.default_rng()
    all_pats = labels_df['PatientID'].tolist()
    overall = lungmets_rate(all_pats, Xc)
    for _ in range(max_tries):
        tr, va, te = split_patients(all_pats, rng=rng)
        if np.isnan(overall) or 'LungMets' not in Xc.columns:
            return tr, va, te
        rates = [lungmets_rate(p, Xc) for p in (tr, va, te)]
        if all(abs(r - overall) <= tol for r in rates):
            return tr, va, te
    return tr, va, te  # last attempt

## CoxPH: prune/align, tuning, and repeated evaluation

In [ ]:
# === CoxPH: prune + align + tuning (ridge-only por padrão) ===
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter

def prune_and_align_for_cox(df_train, df_val, df_test,
                            drop_nan=True, min_pos=1, min_neg=1,
                            keep_cols=('Age','Sex_Male','LungMets','Grade_ord')):
    """
    - Converte dummies booleanas para int.
    - Remove *_nan se drop_nan=True.
    - Mantém apenas colunas com pelo menos `min_pos` 1s e `min_neg` 0s no TRAIN.
    - Para famílias one-hot (prefixo antes do primeiro '_'), remove 1 ref por família (a menos que esteja em keep_cols).
    - Remove colunas duplicadas.
    - Alinha VAL/TEST para ter exatamente as colunas finais do TRAIN.
    """
    dfs = {'train': df_train.copy(), 'val': df_val.copy(), 'test': df_test.copy()}
    for d in dfs.values():
        for c in d.columns:
            if d[c].dtype == bool:
                d[c] = d[c].astype(int)

    tr = dfs['train']
    feat_cols = [c for c in tr.columns if c not in ('PatientID','PFS_time_days','PFS_event')]
    Xtr = tr[feat_cols].copy()
    ytr = tr[['PFS_time_days','PFS_event']].copy()
    n = len(Xtr)

    if drop_nan:
        Xtr = Xtr[[c for c in Xtr.columns if not c.lower().endswith('_nan')]]

    # Filtro por raridade/falta de variação
    sums = Xtr.sum(axis=0)
    keep = sums[(sums >= min_pos) & ((n - sums) >= min_neg)].index.tolist()
    for k in keep_cols:
        if k in Xtr.columns and k not in keep:
            keep.append(k)
    Xtr = Xtr[keep]

    # Remover 1 referência por família one-hot (exceto keep_cols)
    groups = {}
    for c in Xtr.columns:
        if '_' in c and c not in keep_cols:
            base = c.split('_')[0]
            groups.setdefault(base, []).append(c)

    drop_ref = []
    for base, cols in groups.items():
        if len(cols) > 1:
            freqs = Xtr[cols].sum().sort_values()
            for cc in freqs.index:
                if cc not in keep_cols:
                    drop_ref.append(cc)
                    break
    if drop_ref:
        Xtr = Xtr.drop(columns=drop_ref, errors='ignore')

    # Remover colunas duplicadas
    Xtr = Xtr.loc[:, ~Xtr.T.duplicated()]
    final_cols = Xtr.columns.tolist()

    def align(df):
        X = df[[c for c in df.columns if c not in ('PatientID','PFS_time_days','PFS_event')]].copy()
        for c in X.columns:
            if X[c].dtype == bool:
                X[c] = X[c].astype(int)
        if drop_nan:
            X = X[[c for c in X.columns if not c.lower().endswith('_nan')]]
        for c in final_cols:
            if c not in X.columns:
                X[c] = 0
        X = X[final_cols]
        return pd.concat([X, df[['PFS_time_days','PFS_event']]], axis=1)

    tr_final = pd.concat([Xtr, ytr], axis=1)
    va_final = align(dfs['val'])
    te_final = align(dfs['test'])
    return tr_final, va_final, te_final, final_cols, drop_ref


def tune_cox(train_df, val_df, cols, ridge_only=True):
    """
    Faz grid simples de penalizer (e L1 se ridge_only=False) e escolhe pelo C-index na VAL.
    """
    if ridge_only:
        penalizers = (0.01, 0.1, 0.5, 1.0, 2.0, 5.0)
        l1s = (0.0,)
    else:
        penalizers = (0.0, 0.01, 0.1, 0.5, 1.0)
        l1s = (0.0, 0.25, 0.5, 0.75)

    best = (-np.inf, None, None, None)
    Xy_tr = train_df[cols + ['PFS_time_days','PFS_event']]
    Xy_va = val_df[cols + ['PFS_time_days','PFS_event']]

    for pen in penalizers:
        for l1 in l1s:
            try:
                cph = CoxPHFitter(penalizer=pen, l1_ratio=l1)
            except TypeError:
                if l1 != 0.0:
                    continue
                cph = CoxPHFitter(penalizer=pen)
            try:
                cph.fit(Xy_tr, duration_col='PFS_time_days', event_col='PFS_event')
                val_c = cph.score(Xy_va, scoring_method='concordance_index')
                if val_c > best[0]:
                    best = (val_c, pen, l1, cph)
            except Exception:
                continue

    if best[3] is None:
        raise RuntimeError('Penalty tuning failed.')
    print(f"Best Cox | Val C-index={best[0]:.3f} | penalizer={best[1]} | l1_ratio={best[2]}")
    return best[3]


def fit_eval_cox(train_df, val_df, test_df, cols):
    """
    Ajusta com melhor penalidade (val C-index) e retorna (modelo, val_c, test_c).
    """
    model = tune_cox(train_df, val_df, cols, ridge_only=True)
    val_c  = model.score(val_df[cols + ['PFS_time_days','PFS_event']],  scoring_method='concordance_index')
    test_c = model.score(test_df[cols + ['PFS_time_days','PFS_event']], scoring_method='concordance_index')
    return model, float(val_c), float(test_c)


def eval_once_cox(labels_df, X_base, X_rad=None, rng=None, balanced=True, tol=0.15):
    """
    Faz um split por paciente (opcionalmente balanceado em LungMets), monta datasets,
    aplica prune+align e avalia um ajuste do Cox.
    Requer: balanced_split(...) e build_dataset(...) definidos em células anteriores.
    """
    rng = rng or np.random.default_rng()
    if balanced:
        tr, va, te = balanced_split(labels_df, X_base, tol=tol, rng=rng)
    else:
        tr, va, te = split_patients(labels_df['PatientID'], rng=rng)

    dtr = build_dataset(X_base, labels_df, tr, X_rad)
    dva = build_dataset(X_base, labels_df, va, X_rad)
    dte = build_dataset(X_base, labels_df, te, X_rad)

    tr_f, va_f, te_f, cols_f, dropped = prune_and_align_for_cox(dtr, dva, dte)
    # opcional: print de colunas dropadas
    # print("Dropped (ref/rare):", dropped)

    model, v, t = fit_eval_cox(tr_f, va_f, te_f, cols_f)
    return v, t


def repeated_eval_cox(X_base, R=30, use_radiomics=False, X_rad=None,
                      balanced=True, tol=0.15, seed=123):
    """
    Repete R vezes o split/treino/val/test e retorna arrays com C-index de VAL e TEST.
    """
    rng = np.random.default_rng(seed)
    vals, tests = [], []
    for _ in range(R):
        v, t = eval_once_cox(labels, X_base,
                             X_rad if use_radiomics else None,
                             rng=rng, balanced=balanced, tol=tol)
        vals.append(v)
        tests.append(t)
    return np.array(vals), np.array(tests)


## DeepSurv (pycox): model, training, repeated evaluation

In [ ]:
def to_deep_tensors(df):
    feats = [c for c in df.columns if c not in ('PatientID','PFS_time_days','PFS_event')]
    x = df[feats].values.astype('float32')
    y_time  = df['PFS_time_days'].values.astype('float32')
    y_event = df['PFS_event'].values.astype('int64')
    return x, y_time, y_event, feats

def make_deepsurv_net(in_features):
    return torch.nn.Sequential(
        torch.nn.Linear(in_features, 32),
        torch.nn.ReLU(),
        torch.nn.BatchNorm1d(32),
        torch.nn.Dropout(0.1),
        torch.nn.Linear(32, 1)
    )

def train_deepsurv(train_df, val_df):
    x_tr,t_tr,e_tr,feats = to_deep_tensors(train_df)
    x_va,t_va,e_va,_     = to_deep_tensors(val_df)
    net = make_deepsurv_net(x_tr.shape[1])
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = DeepCoxPH(net, tt.optim.Adam(lr=1e-3), device=device)
    callbacks = [tt.callbacks.EarlyStopping(patience=40)]
    model.fit(x_tr, (t_tr, e_tr), batch_size=16, epochs=400,
              val_data=(x_va,(t_va,e_va)), callbacks=callbacks, verbose=False)
    return model, feats

def deep_concordance(model, df, feats):
    x,t,e,_ = to_deep_tensors(df[feats + ['PFS_time_days','PFS_event']].copy())
    # predict risk (higher is worse); use partial hazards via network output
    with torch.no_grad():
        risk = model.net(torch.tensor(x)).cpu().numpy().ravel()
    return float(concordance_index(df['PFS_time_days'], -risk, df['PFS_event']))

def eval_once_deep(labels_df, X_base, X_rad=None, rng=None, balanced=True, tol=0.15):
    rng = rng or np.random.default_rng()
    tr, va, te = balanced_split(labels_df, X_base, tol=tol, rng=rng) if balanced else split_patients(labels_df['PatientID'], rng=rng)
    dtr = build_dataset(X_base, labels_df, tr, X_rad)
    dva = build_dataset(X_base, labels_df, va, X_rad)
    dte = build_dataset(X_base, labels_df, te, X_rad)
    # ensure numeric only
    for df in (dtr,dva,dte):
        for c in df.columns:
            if df[c].dtype == bool: df[c] = df[c].astype(int)
    model, feats = train_deepsurv(dtr, dva)
    v = deep_concordance(model, dva, feats)
    t = deep_concordance(model, dte, feats)
    return v, t

def repeated_eval_deep(X_base, R=30, use_radiomics=False, X_rad=None,
                       balanced=True, tol=0.15, seed=123):
    rng = np.random.default_rng(seed)
    vals, tests = [], []
    for _ in range(R):
        v, t = eval_once_deep(labels, X_base, X_rad if use_radiomics else None, rng=rng, balanced=balanced, tol=tol)
        vals.append(v); tests.append(t)
    return np.array(vals), np.array(tests)

## Run experiments (CoxPH & DeepSurv) and print two SUMMARY blocks

In [ ]:
# === Rodar experimentos e imprimir SUMÁRIO ===

# CoxPH
cox_vals_clin, cox_tests_clin = repeated_eval_cox(
    X_clin_cox, R=REPEATS_R, use_radiomics=False,
    X_rad=X_rad, balanced=BALANCED_REPEATS,
    tol=BALANCE_TOL, seed=123
)

if X_rad is not None:
    # Radiomics only: usa apenas PatientID e insere radiomics dentro do pipeline
    X_empty = pd.DataFrame({'PatientID': X_rad['PatientID']})
    cox_vals_rad,  cox_tests_rad  = repeated_eval_cox(
        X_empty, R=REPEATS_R, use_radiomics=True,
        X_rad=X_rad, balanced=BALANCED_REPEATS,
        tol=BALANCE_TOL, seed=321
    )
    # Clinical + Radiomics
    cox_vals_comb, cox_tests_comb = repeated_eval_cox(
        X_clin_cox, R=REPEATS_R, use_radiomics=True,
        X_rad=X_rad, balanced=BALANCED_REPEATS,
        tol=BALANCE_TOL, seed=555
    )
else:
    cox_vals_rad = cox_tests_rad = np.array([])
    cox_vals_comb = cox_tests_comb = np.array([])

# DeepSurv
deep_vals_clin, deep_tests_clin = repeated_eval_deep(
    X_clin_deep, R=REPEATS_R, use_radiomics=False,
    X_rad=X_rad, balanced=BALANCED_REPEATS,
    tol=BALANCE_TOL, seed=123
)

if X_rad is not None:
    X_empty = pd.DataFrame({'PatientID': X_rad['PatientID']})
    deep_vals_rad,  deep_tests_rad  = repeated_eval_deep(
        X_empty, R=REPEATS_R, use_radiomics=True,
        X_rad=X_rad, balanced=BALANCED_REPEATS,
        tol=BALANCE_TOL, seed=321
    )
    deep_vals_comb, deep_tests_comb = repeated_eval_deep(
        X_clin_deep, R=REPEATS_R, use_radiomics=True,
        X_rad=X_rad, balanced=BALANCED_REPEATS,
        tol=BALANCE_TOL, seed=555
    )
else:
    deep_vals_rad = deep_tests_rad = np.array([])
    deep_vals_comb = deep_tests_comb = np.array([])

def _fmt(m, s):
    return f"{m:.3f} ± {s:.3f}"

print("\n=== Cox SUMMARY (mean ± sd, Test C-index) ===")
print("Clinical only:          ", _fmt(cox_tests_clin.mean(), cox_tests_clin.std()))
if len(cox_tests_rad):
    print("Radiomics only (median):", _fmt(cox_tests_rad.mean(),  cox_tests_rad.std()))
    print("Clinical + radiomics:   ", _fmt(cox_tests_comb.mean(), cox_tests_comb.std()))
else:
    print("Radiomics only (median): N/A")
    print("Clinical + radiomics:    N/A")

print("\n=== DeepSurv SUMMARY (mean ± sd, Test C-index) ===")
print("Clinical only:          ", _fmt(deep_tests_clin.mean(), deep_tests_clin.std()))
if len(deep_tests_rad):
    print("Radiomics only (median):", _fmt(deep_tests_rad.mean(),  deep_tests_rad.std()))
    print("Clinical + radiomics:   ", _fmt(deep_tests_comb.mean(), deep_tests_comb.std()))
else:
    print("Radiomics only (median): N/A")
    print("Clinical + radiomics:    N/A")
